# 02 — Data Cleaning

Goal: produce a single cleaned source-of-truth dataset for analysis and the public dashboard. No rows are removed and no missing values are imputed.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "listings.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "boston_airbnb_cleaned.csv"
raw = pd.read_csv(RAW_PATH, low_memory=False)
raw_sha_before = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw.shape

(3585, 95)

## Deterministic parsers

In [2]:
def parse_currency(series):
    return pd.to_numeric(series.astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

def parse_pct(series):
    return pd.to_numeric(series.astype("string").str.replace("%", "", regex=False), errors="coerce")

## Transform selected fields

In [3]:
work = raw.copy()
work["price_usd"] = parse_currency(work["price"])
work["cleaning_fee_usd"] = parse_currency(work["cleaning_fee"])
work["extra_people_usd"] = parse_currency(work["extra_people"])
work["host_response_rate_pct"] = parse_pct(work["host_response_rate"])
work["host_acceptance_rate_pct"] = parse_pct(work["host_acceptance_rate"])

for col in ["host_is_superhost", "host_identity_verified", "instant_bookable"]:
    work[col] = work[col].map({"t": True, "f": False})

## Select analysis-relevant columns

In [4]:
clean_cols = ['id', 'host_id', 'last_scraped', 'host_since', 'host_response_time', 'host_response_rate_pct', 'host_acceptance_rate_pct', 'host_is_superhost', 'host_identity_verified', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'bed_type', 'price_usd', 'cleaning_fee_usd', 'guests_included', 'extra_people_usd', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'cancellation_policy', 'calculated_host_listings_count', 'reviews_per_month']
cleaned = work[clean_cols].copy()
cleaned.shape

(3585, 43)

## Validation before save

In [5]:
assert len(cleaned) == len(raw), "Row count changed"
assert cleaned["id"].is_unique, "Listing ID is no longer unique"
assert cleaned["price_usd"].notna().all(), "A nightly price failed to parse"
assert ((raw["price"].notna()) & parse_currency(raw["price"]).isna()).sum() == 0
assert ((raw["cleaning_fee"].notna()) & parse_currency(raw["cleaning_fee"]).isna()).sum() == 0
assert ((raw["extra_people"].notna()) & parse_currency(raw["extra_people"]).isna()).sum() == 0
assert ((raw["host_response_rate"].notna()) & parse_pct(raw["host_response_rate"]).isna()).sum() == 0
assert ((raw["host_acceptance_rate"].notna()) & parse_pct(raw["host_acceptance_rate"]).isna()).sum() == 0

print("Raw rows:", len(raw))
print("Cleaned rows:", len(cleaned))
print("Rows removed:", len(raw) - len(cleaned))
print("Cleaned columns:", cleaned.shape[1])

Raw rows: 3585
Cleaned rows: 3585
Rows removed: 0
Cleaned columns: 43


## Save UTF-8 cleaned CSV

In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(OUTPUT_PATH)

/mnt/data/boston-airbnb-listings-analysis/data/processed/boston_airbnb_cleaned.csv


## Post-save integrity

In [7]:
reloaded = pd.read_csv(OUTPUT_PATH)
raw_sha_after = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
assert raw_sha_before == raw_sha_after, "Raw file changed"
assert reloaded.shape == cleaned.shape
assert reloaded["id"].tolist() == cleaned["id"].tolist()
print("Raw SHA-256 unchanged:", raw_sha_after)
print("Processed shape:", reloaded.shape)

Raw SHA-256 unchanged: 9c2fa5db20b0d1018cb558750aeb276599ea877e1e0c1ae315766a032d78881f
Processed shape: (3585, 43)


## Cleaning result

The processed file contains the same 3,585 listings as the source. The project deliberately preserves nulls and high-price observations rather than making unsupported corrections.